# Hypoxia signature model — single-dataset discovery + scorer\n\nBuilt incrementally per `CLAUDE.md`. Scope for this session: seed list → discovery (Eqs 1, 2, 5 + Monte Carlo) → scorer (Eq 7 + HS). Adapters, Cox validation, and sigQC are out of scope here."

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats

## Seed genes (Buffa 2010, HNSCC training network, set A)

The ten seed genes exactly as printed in the paper, hardcoded — not a file to
load. One of the ten (`AK3L1`) was later renamed by HGNC to `AK4`; modern
datasets use the new symbol, so we keep both forms and resolve at lookup time
against whatever gene index the actual dataset has.

In [ ]:
# Ten seed genes, literal names as printed in Buffa et al. 2010.
SEED_GENES = [
    "ADM", "AK3L1", "BNIP3", "CA9", "ENO1",
    "HK2", "LDHA", "PGK1", "SLC2A1", "VEGFA",
]

# AK3L1 -> AK4 was a formal HGNC rename, not a casual alias. Exact-string
# matching against a modern dataset (Ensembl/GEO/TCGA-annotated) will miss
# AK3L1 entirely and silently drop that seed unless we also try AK4.
SEED_ALIASES = {
    "AK3L1": "AK4",
}


def resolve_seed_genes(seed_genes: list[str], available_genes) -> dict[str, str]:
    """Map each seed (paper name) to whichever symbol is present in `available_genes`.

    Tries the literal paper name first, then its known alias. Seeds matching
    neither are left out of the returned dict -- callers should check for
    missing seeds rather than assume all ten resolve.
    """
    available = set(available_genes)
    resolved = {}
    for seed in seed_genes:
        if seed in available:
            resolved[seed] = seed
        elif seed in SEED_ALIASES and SEED_ALIASES[seed] in available:
            resolved[seed] = SEED_ALIASES[seed]
    return resolved

## Discovery step 1: seed–gene affinity + membership (Eqs 1, 2 — exact formulas)

Replacing the reconstructed version from before with the paper's actual
equations, now that we have them.

**Eq 1 — affinity, d(p_i, y_j):**

d(p_i, y_j) = [1 + exp(-(r²(p_i,y_j) - γ_t) / γ_s)]^-1

A sigmoid soft-threshold on `r²` (squared Spearman correlation between seed
`p_i` and gene `y_j`). `γ_t` is the cluster boundary — a Bonferroni-corrected
(α = 0.05) significance threshold on `r²` — and `γ_s` controls the sigmoid's
sharpness. As `γ_s → 0` (the form the paper actually uses in this study),
this collapses to a hard step: `d = 1` if `r² > γ_t`, else `0`. Implemented
below with `gamma_s=0` as the default (hard step); the literal sigmoid is
available by passing a small positive `gamma_s`.

This also resolves the ambiguity flagged in the previous version of this
chunk: `γ_t` is explicitly a threshold *on `r²`*, not on `|r|`. So
`critical_r_squared()` below returns the critical value already squared —
no more guessing which scale it's on.

**Eq 2 — membership, g(y_i, p_k):**

g(y_i, p_k) = d(y_i, p_k) / Σ_{j=1}^{K} d(y_i, p_j)

This is genuinely different from the previous chunk's `gamma = delta *
|rho|`, which was a guess at "increases with `|ρ|`" — wrong, now that the
real formula is available. Eq 2 normalizes gene `y_i`'s affinity to seed
`p_k` against its affinity to *all* `K` seeds combined, so it needs every
seed's affinity for a gene computed together, not one seed handled in
isolation. That's why the functions below build a genes × seeds affinity
matrix first, then normalize each gene's row across seeds.

In [ ]:
def critical_r_squared(n: int, m: int, alpha: float = 0.05) -> float:
    """gamma_t in Eq 1: critical r^2 for significance at `alpha`,
    Bonferroni-corrected across `m` comparisons, for a sample of size `n`.

    Finds the critical t-statistic (two-tailed, df = n - 2) for the
    corrected alpha, converts it to a critical |r| via the standard
    r <-> t relationship (t = r * sqrt((n - 2) / (1 - r^2))), then squares
    it, since Eq 1 thresholds r^2 directly.
    """
    alpha_corrected = alpha / m
    df = n - 2
    t_crit = stats.t.ppf(1 - alpha_corrected / 2, df)
    r_crit = t_crit / np.sqrt(df + t_crit ** 2)
    return r_crit ** 2


def seed_gene_correlations(matrix: pd.DataFrame, seed_symbols: list[str]) -> pd.DataFrame:
    """Spearman rho between every gene (column of `matrix`) and each seed gene.

    matrix       : samples x genes matrix
    seed_symbols : gene symbols to correlate against, as they appear in
                   matrix.columns (i.e. resolve_seed_genes(...).values())

    Returns a genes x seeds DataFrame of rho.

    Computed as rank-transform once, then Pearson-correlate the ranks via
    matrix algebra (Spearman's rho is just Pearson's r on ranks) -- NOT via
    pandas' matrix.corrwith(other, method="spearman") in a per-seed loop.
    That per-column approach re-ranks the whole matrix on every call and
    turned out ~500-600x slower in testing (2.9s for 10 seeds x 525 genes
    vs 0.004s here) -- fatal once this runs inside the Monte Carlo loop,
    which calls it hundreds of times.
    """
    ranked = matrix.rank()
    seed_ranked = ranked[seed_symbols]
    n = len(matrix)
    gene_centered = ranked - ranked.mean()
    seed_centered = seed_ranked - seed_ranked.mean()
    cov = gene_centered.T.dot(seed_centered) / n
    gene_std = ranked.std(ddof=0)
    seed_std = seed_ranked.std(ddof=0)
    return cov.div(gene_std, axis=0).div(seed_std, axis=1)


def affinity(rho: pd.DataFrame, gamma_t: float, gamma_s: float = 0.0) -> pd.DataFrame:
    """Eq 1: d(p_i, y_j) -- seed-gene affinity, genes x seeds.

    rho     : genes x seeds Spearman correlations (seed_gene_correlations())
    gamma_t : critical r^2 threshold (critical_r_squared())
    gamma_s : sigmoid sharpness. 0 (default) -> hard step, the gamma_s -> 0
              limit the paper uses in this study. A small positive value
              gives the literal sigmoid instead.
    """
    r2 = rho ** 2
    if gamma_s == 0:
        return (r2 > gamma_t).astype(float)
    return 1.0 / (1.0 + np.exp(-(r2 - gamma_t) / gamma_s))


def membership(d: pd.DataFrame) -> pd.DataFrame:
    """Eq 2: g(y_i, p_k) = d(y_i, p_k) / sum_j d(y_i, p_j).

    Normalizes each gene's row of affinities across all K seeds. Genes with
    zero affinity to every seed (the common case) hit 0/0 -- defined here as
    0 for all seeds, not NaN.
    """
    row_sums = d.sum(axis=1)
    g = d.div(row_sums, axis=0)
    return g.fillna(0.0)

## Discovery step 2: connectivity score (Eq 5)

C(y_i) = [ Σ_{j=1,j≠i}^{K} w(p_j) g(y_i, p_j) ] / [ Σ_{h=1,h≠i}^{K} w(p_h) ]

For each gene `y_i`, average its membership `g(y_i, p_j)` across all `K`
seeds — the "hub score." `w(p_j) = 1` normally, `0` if `y_i` **is** that seed
itself (a seed's own column in the membership matrix), so a seed doesn't get
inflated connectivity purely from being perfectly correlated with itself.
Since `w` only ever excludes at most one column (a gene can be at most one
of the seeds), this is just: mean over all `K` seed columns, except for the
`K` genes that are themselves seeds, where it's the mean over the other
`K - 1` columns.

`C` is then converted to a fractional rank in `[0, 1]` — this is the score
genes get ranked and thresholded on downstream (via the Monte Carlo step,
next chunk).

In [ ]:
def connectivity(g: pd.DataFrame) -> pd.Series:
    """Eq 5: C(y_i) -- connectivity score, fractional-ranked.

    g : genes x seeds membership matrix (membership()). Columns are seed
        symbols as they appear in the dataset (i.e. resolve_seed_genes()
        values), so a gene that IS a seed matches its own column name
        directly -- no separate seed-name lookup needed.

    For each gene, averages g across all K seed columns, except that a gene
    which is itself one of the seeds excludes its own column from both the
    sum and the count (w = 0 for self, per Eq 5). Returns C as a fractional
    rank in [0, 1] across genes.
    """
    K = g.shape[1]
    row_sum = g.sum(axis=1)
    C_raw = row_sum / K

    for seed_col in g.columns:
        if seed_col in C_raw.index:
            C_raw[seed_col] = (row_sum[seed_col] - g.loc[seed_col, seed_col]) / (K - 1)

    return C_raw.rank(pct=True)

## Discovery step 3: Monte Carlo null distribution + significance threshold

Not one of the paper's numbered equations, but explicit in the Methods and
required for single-dataset discovery (see CLAUDE.md's Monte Carlo vs
bootstrap note): build many networks from **random** fake seed sets (same
size `K` as the real seed set) to see how high a gene's connectivity `C` can
get by chance alone, then set the inclusion threshold from that null
distribution.

Procedure, repeated `n_monte_carlo` times:
1. Pick `K` random genes from the dataset as a fake seed set.
2. Run the *same* Eq 1 → Eq 2 → Eq 5 pipeline on that fake seed set to get a
   null `C` for every gene.
3. Pool every gene's null `C` from every iteration into one big distribution.

The threshold is the `(1 - alpha)` quantile of that pooled null distribution
— e.g. the 99th percentile at `alpha = 0.01`. A gene from the *real* seed
network is kept as a significant hub if its real `C` exceeds this threshold.
This is a single scalar threshold on `C`, matching CLAUDE.md's framing
("Monte-Carlo simulations used to determine threshold of C for inclusion"),
not a per-gene p-value.

`gamma_t` (the Bonferroni threshold) doesn't need recomputing per iteration
— it depends only on sample size `n` and total genes tested `m`, both fixed
regardless of which genes happen to be picked as fake seeds.

Heads-up on runtime: each iteration recomputes correlations for `K` genes
against the whole matrix, `n_monte_carlo` times — with 1000 iterations and a
real-sized gene matrix this can take a while. Worth testing with a smaller
`n_monte_carlo` (e.g. 100) first to confirm it runs, before scaling up.

In [ ]:
def monte_carlo_threshold(matrix: pd.DataFrame, k_seeds: int, gamma_t: float,
                           n_monte_carlo: int = 1000, alpha: float = 0.01,
                           gamma_s: float = 0.0, random_state=None) -> float:
    """Null distribution for C (Eq 5) from random fake seed sets, and the
    resulting significance threshold.

    matrix        : samples x genes matrix
    k_seeds       : size of the fake seed sets to draw (K, same as the real
                    seed count)
    gamma_t       : critical r^2 threshold (critical_r_squared()) -- same
                    value used for the real network, since it only depends
                    on n (samples) and m (genes tested), not on which genes
                    are seeds
    n_monte_carlo : number of random fake-seed networks to build
    alpha         : significance level; threshold = (1 - alpha) quantile of
                    the pooled null C distribution
    random_state  : seed for reproducibility (passed to numpy's Generator)

    Returns a single scalar threshold: real genes with C above this are
    treated as significant hubs.
    """
    rng = np.random.default_rng(random_state)
    genes = matrix.columns.to_numpy()
    null_C = []

    for _ in range(n_monte_carlo):
        fake_seeds = rng.choice(genes, size=k_seeds, replace=False)
        rho_null = seed_gene_correlations(matrix, fake_seeds)
        d_null = affinity(rho_null, gamma_t, gamma_s)
        g_null = membership(d_null)
        C_null = connectivity(g_null)
        null_C.append(C_null.to_numpy())

    null_C = np.concatenate(null_C)
    return float(np.quantile(null_C, 1 - alpha))


def significant_hub_genes(C_real: pd.Series, threshold: float) -> list[str]:
    """Genes whose real connectivity C exceeds the Monte Carlo threshold."""
    return C_real[C_real > threshold].index.tolist()

## Scorer: expression score E (Eq 7) + Hypoxia Score (HS)

E(sample) = median( expression(gene, sample) )  over signature genes

Median, not mean, is deliberate: robust to outlier genes, and degrades
gracefully when some signature genes are missing from a sample's platform
(the key requirement for Olink later, which won't cover every gene in the
list) — a few missing genes just shrink the set the median is taken over,
rather than breaking the score.

HS(sample) = fractional_rank( E(sample) )  across samples in the same cohort,
rescaled to [0, 1]

`E` and `HS` are kept as two separate functions rather than one, because
CLAUDE.md flags that HS is cohort-relative by construction — with a cohort
of 2 (the pooled normoxic/hypoxic samples expected Friday), ranking two
things against each other is close to meaningless, and the raw `E` values
should be compared directly instead. Keeping `E` available on its own is
what makes that comparison possible; if `hypoxia_score()` folded `E` inside
and only returned `HS`, that use case would be lost.

This is also the platform-agnostic core CLAUDE.md's architecture section
describes: everything from here on only needs a samples × genes matrix of
comparable values — same functions for RNA-seq and Olink alike, which is
what makes "does the Olink score match the RNA-seq score" a meaningful
question later.

In [ ]:
def expression_score(matrix: pd.DataFrame, signature_genes: list[str]) -> pd.Series:
    """Eq 7: E(sample) = median expression across signature genes present in `matrix`.

    matrix          : samples x genes matrix (any platform, already on a
                      comparable scale -- log2(TPM+1), NPX, etc.)
    signature_genes : gene list to score with (e.g. significant_hub_genes()
                      output for this dataset)

    Genes in `signature_genes` not present in `matrix.columns` are silently
    dropped rather than erroring -- this is what lets Olink (partial gene
    coverage) use the same function as RNA-seq.
    """
    available = [g for g in signature_genes if g in matrix.columns]
    return matrix[available].median(axis=1)


def hypoxia_score(E: pd.Series) -> pd.Series:
    """HS(sample) = fractional rank of E within this cohort, rescaled to [0, 1].

    Cohort-relative by construction: ranks samples only against each other
    in the same `E`. Meaningless for very small cohorts (e.g. n=2) -- compare
    raw E values directly in that case instead of HS.
    """
    return E.rank(pct=True)

## Sanity check: synthetic dataset (test-only, not adapter/production code)

Before pointing this at real data, validate the whole chain
(seeds → affinity → membership → connectivity → Monte Carlo → scorer) on a
toy dataset where we know the right answer by construction.

Design, and why it looks like this (the first version I tried was wrong —
worth explaining, since it's a real pitfall, not just a style choice):

- One latent "hypoxia" factor drives the 10 seed genes plus `n_hub_genes`
  more genes — these are the genes discovery *should* recover.
- **Effect sizes are staggered** (`rng.uniform` over a range), not identical
  for every hub gene. My first attempt used one fixed effect size for all
  hub genes, which made them near-perfectly tied in real `C` — and
  `rank(pct=True)` averages tied ranks, which pinned the real hub cluster's
  score almost exactly at the Monte Carlo threshold. Result: 0 genes
  recovered, not because the code was wrong, but because the test was an
  edge case by construction. Staggering effect sizes avoids that coincidence.
- A second, **independent "distractor" module** (`n_distractor_genes`,
  driven by an unrelated latent factor) stands in for the many unrelated
  co-expression modules a real transcriptome has — genes that are strongly
  correlated with *each other* but not with the real seeds. Without this,
  the only strongly-correlated block in the whole dataset is the real hub
  cluster itself, so the Monte Carlo null (built from random fake seeds)
  gets contaminated almost entirely by hits on the real signal, which
  inflated the threshold artificially in the first version.
- The rest (`n_noise_genes`) is pure independent noise.
- `n_noise_genes=2000` (not 500) also matters: it keeps the true hub
  fraction small relative to the whole gene universe, closer to a real
  dataset's proportions, rather than ~5% of all genes being "hot."

Also uses `AK4` (not `AK3L1`) as the column name for that one seed, to
exercise the alias-resolution path from `resolve_seed_genes()` end to end.

In [ ]:
def make_synthetic_dataset(n_samples=60, n_hub_genes=15, n_distractor_genes=30,
                            n_noise_genes=2000, seed_names=None,
                            hub_effect_range=(1.5, 4.0), distractor_effect_range=(1.5, 4.0),
                            noise_sd=1.0, random_state=0) -> pd.DataFrame:
    """Toy samples x genes matrix for sanity-checking the discovery pipeline.

    Four groups of genes:
      - the 10 seed genes + `n_hub_genes` more, all driven by one latent
        "hypoxia" factor, each at its own random effect size in
        `hub_effect_range` (staggered on purpose -- see markdown above)
      - `n_distractor_genes` driven by a second, independent latent factor
        (an unrelated co-expression module)
      - `n_noise_genes` pure noise, correlated with nothing

    Returns a samples x genes DataFrame.
    """
    rng = np.random.default_rng(random_state)
    if seed_names is None:
        seed_names = ["ADM", "AK4", "BNIP3", "CA9", "ENO1",
                      "HK2", "LDHA", "PGK1", "SLC2A1", "VEGFA"]

    hypoxia_factor = rng.normal(size=n_samples)
    distractor_factor = rng.normal(size=n_samples)

    columns = {}
    for gene in seed_names:
        effect = rng.uniform(*hub_effect_range)
        columns[gene] = effect * hypoxia_factor + rng.normal(scale=noise_sd, size=n_samples)
    for i in range(n_hub_genes):
        effect = rng.uniform(*hub_effect_range)
        columns[f"HUB{i}"] = effect * hypoxia_factor + rng.normal(scale=noise_sd, size=n_samples)
    for i in range(n_distractor_genes):
        effect = rng.uniform(*distractor_effect_range)
        columns[f"DISTRACTOR{i}"] = effect * distractor_factor + rng.normal(scale=noise_sd, size=n_samples)
    for i in range(n_noise_genes):
        columns[f"NOISE{i}"] = rng.normal(scale=noise_sd, size=n_samples)

    return pd.DataFrame(columns)


matrix = make_synthetic_dataset(random_state=0)
print("matrix shape:", matrix.shape)

resolved = resolve_seed_genes(SEED_GENES, matrix.columns)
seed_symbols = list(resolved.values())
print(f"Resolved {len(seed_symbols)}/{len(SEED_GENES)} seeds:", resolved)

n, m = matrix.shape
gamma_t = critical_r_squared(n=n, m=m, alpha=0.05)

rho = seed_gene_correlations(matrix, seed_symbols)
d = affinity(rho, gamma_t)
g = membership(d)
C = connectivity(g)

threshold = monte_carlo_threshold(matrix, k_seeds=len(seed_symbols), gamma_t=gamma_t,
                                   n_monte_carlo=300, alpha=0.01, random_state=1)
hub_genes = significant_hub_genes(C, threshold)

expected_hub_like = set(seed_symbols) | {f"HUB{i}" for i in range(15)}
found = set(hub_genes)
print(f"Monte Carlo threshold: {threshold:.4f}")
print(f"Derived signature ({len(hub_genes)} genes): {sorted(hub_genes)}")
print(f"True positives: {len(found & expected_hub_like)}/{len(expected_hub_like)} hub-like genes recovered")
print(f"False positives: {len(found - expected_hub_like)} (should be small -- alpha={0.01} on ~{m} genes)")
print(f"Missed: {sorted(expected_hub_like - found)}")

E = expression_score(matrix, hub_genes)
HS = hypoxia_score(E)
print("\nE (expression score):\n", E.describe())